# Finnish Financial Sent Template Notebook

## Imports

In [1]:
from datetime import datetime

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score,
)

## Enable if upsampling is used 
# from sklearn.utils import resample 
from sklearn.model_selection import train_test_split
from datasets import Dataset
import pandas as pd
import datetime
import numpy as np
import torch

## Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## GPU/RAM Check

In [3]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Fri Mar 20 06:32:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          Off |   00000000:04:00.0 Off |                    0 |
| N/A   28C    P0             67W /  700W |       4MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
import psutil

ram_gb = psutil.virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

Your runtime has 247.0 gigabytes of available RAM

You are using a high-RAM runtime!


## Load Model

In [5]:
timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

BASE_MODEL = 'FacebookAI/xlm-roberta-large'
FINNSENTIMENT_MODEL_PATH = f'/content/drive/MyDrive/ColabThesisData/{BASE_MODEL.split("/")[-1]}_finnsentiment_{timestamp}/'
NUM_LABELS  = 3
LABEL_NAMES = ["negative", "neutral", "positive"]
MAX_SEQ_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=NUM_LABELS,
    device_map="auto",
    dtype=torch.bfloat16,
)

print(f"Base model loaded: {BASE_MODEL}")
print(f"Model device: {next(model.parameters()).device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Base model loaded: FacebookAI/xlm-roberta-large
Model device: cuda:0


## Define Eval Func

In [6]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.argmax(preds, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted", zero_division=0),
        "precision": precision_score(labels, preds, average="weighted", zero_division=0),
        "recall": recall_score(labels, preds, average="weighted", zero_division=0),
    }

## Domain-adapted pretraining

In [7]:
from transformers import AutoModelForMaskedLM, DataCollatorForLanguageModeling

DAPT_SAMPLE_SEED = 42
DAPT_N_SAMPLES   = 50_000
DAPT_MODEL_PATH  = f'/content/drive/MyDrive/ColabThesisData/{BASE_MODEL.split("/")[-1]}_dapt_{timestamp}/'

# ── Sample unlabeled forum posts ──────────────────────────────────────────────
forum_posts = pd.read_parquet('/content/drive/MyDrive/ColabThesisData/cleaned_forum_posts.parquet')
dapt_df = (
    forum_posts[['message']]
    .dropna(subset=['message'])
    .sample(n=DAPT_N_SAMPLES, random_state=DAPT_SAMPLE_SEED)
    .reset_index(drop=True)
)
print(f"DAPT corpus: {len(dapt_df):,} posts (seed={DAPT_SAMPLE_SEED})")

# ── Load base model for masked language modelling ─────────────────────────────
# Must use AutoModelForMaskedLM — the classification head is irrelevant for DAPT
dapt_model = AutoModelForMaskedLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    dtype=torch.bfloat16,
)

# ── Tokenize ──────────────────────────────────────────────────────────────────
dapt_ds = Dataset.from_pandas(dapt_df)
dapt_ds = dapt_ds.map(
    lambda batch: tokenizer(batch['message'], truncation=True, max_length=MAX_SEQ_LENGTH),
    batched=True,
    remove_columns=['message'],
)

# ── Train ─────────────────────────────────────────────────────────────────────
dapt_training_args = TrainingArguments(
    output_dir='/tmp/dapt_checkpoints/',
    num_train_epochs=1,          # 1 epoch over 50k posts is standard for DAPT
    per_device_train_batch_size=16,
    learning_rate=1e-4,          # higher than fine-tuning — this is continued pre-training
    weight_decay=0.01,
    warmup_ratio=0.06,
    logging_steps=200,
    save_strategy="epoch",
    save_total_limit=1,
    bf16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=True,
)

dapt_trainer = Trainer(
    model=dapt_model,
    args=dapt_training_args,
    train_dataset=dapt_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=True,
        mlm_probability=0.15,   # standard BERT masking rate
    ),
)

dapt_trainer.train()

dapt_trainer.save_model(DAPT_MODEL_PATH)
tokenizer.save_pretrained(DAPT_MODEL_PATH)
print(f"DAPT model saved to: {DAPT_MODEL_PATH}")

DAPT corpus: 50,000 posts (seed=42)


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

XLMRobertaForMaskedLM LOAD REPORT from: FacebookAI/xlm-roberta-large
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
200,1.845095
400,1.985420
600,1.985682
800,1.948651
1000,1.868116
1200,1.859643
1400,1.799895
1600,1.776512
1800,1.780530
2000,1.733970


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DAPT model saved to: /content/drive/MyDrive/ColabThesisData/xlm-roberta-large_dapt_2026-03-20_06-32-41/


## FinnSentiment Pre-training

In [8]:
finnsentiment = pd.read_pickle('/content/drive/MyDrive/ColabThesisData/finnsentiment_cleaned_unambiguous.pkl')

df_0 = finnsentiment[finnsentiment['label'] == 0]
df_1 = finnsentiment[finnsentiment['label'] == 1]
df_2 = finnsentiment[finnsentiment['label'] == 2]
df_1_balanced = df_1.sample(n=min(len(df_0), len(df_2)), random_state=42)
finnsentiment_balanced = pd.concat([df_0, df_1_balanced, df_2]).reset_index(drop=True)

print(f"FinnSentiment balanced: {len(finnsentiment_balanced):,}")
print(finnsentiment_balanced['label'].value_counts().sort_index())

FinnSentiment balanced: 3,818
label
0    1230
1    1230
2    1358
Name: count, dtype: int64


In [9]:
finnsentiment_balanced.sample(5)

,label,text
2562,2,Ylivoimaisesti osuvin kommentti.
1425,1,"Ja mulla ei koskaan ole ollut pentuja, en oo n..."
2186,1,5.4. 2017 klo 17.30 on Korpitien koululla huol...
3494,2,Kiitos paljon huojentavasta tiedosta!
3046,2,Hei pitkästä aikaa!


In [10]:
finnsentiment_balanced["label"].value_counts()

,count
label,
2,1358
0,1230
1,1230


In [12]:
model = AutoModelForSequenceClassification.from_pretrained(
    DAPT_MODEL_PATH,
    num_labels=NUM_LABELS,
    device_map="auto",
    dtype=torch.bfloat16,
)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/ColabThesisData/xlm-roberta-large_dapt_2026-03-20_06-32-41/
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.argmax(preds, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted", zero_division=0),
        "precision": precision_score(labels, preds, average="weighted", zero_division=0),
        "recall": recall_score(labels, preds, average="weighted", zero_division=0),
    }

# 90/10 train/val split for FinnSentiment
fs_train_df, fs_val_df = train_test_split(
    finnsentiment_balanced, test_size=0.1, random_state=42,
    stratify=finnsentiment_balanced['label'],
)

def make_fs_dataset(df):
    hf = Dataset.from_pandas(df[['text', 'label']].reset_index(drop=True))
    return hf.map(
        lambda batch: tokenizer(batch['text'], truncation=True, max_length=MAX_SEQ_LENGTH),
        batched=True,
    )

fs_train_ds = make_fs_dataset(fs_train_df)
fs_val_ds   = make_fs_dataset(fs_val_df)

fs_training_args = TrainingArguments(
    output_dir='/tmp/fs_checkpoints/',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=75,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    bf16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=True,
)

fs_trainer = Trainer(
    model=model,
    args=fs_training_args,
    train_dataset=fs_train_ds,
    eval_dataset=fs_val_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

fs_trainer.train()

Map:   0%|          | 0/3436 [00:00<?, ? examples/s]

Map:   0%|          | 0/382 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,1.041031,0.450262,0.371861,0.380157,0.450262
2,No log,0.500544,0.790576,0.779758,0.795398,0.790576
3,0.915722,0.346216,0.879581,0.878909,0.880613,0.879581
4,0.915722,0.292324,0.903141,0.902426,0.903110,0.903141
5,0.431959,0.288180,0.897906,0.897207,0.898585,0.897906


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=1075, training_loss=0.6533892236753952, metrics={'train_runtime': 68.0568, 'train_samples_per_second': 252.436, 'train_steps_per_second': 15.796, 'total_flos': 2591695316751216.0, 'train_loss': 0.6533892236753952, 'epoch': 5.0})

In [15]:
fs_trainer.save_model(FINNSENTIMENT_MODEL_PATH)
tokenizer.save_pretrained(FINNSENTIMENT_MODEL_PATH)
print(f"FinnSentiment-tuned model saved to: {FINNSENTIMENT_MODEL_PATH}")

fs_results = fs_trainer.predict(fs_val_ds)
fs_preds = np.argmax(fs_results.predictions, axis=1)
ft_true = fs_val_df["label"].tolist()

print("\n" + "=" * 50)
print("HOLDOUT RESULTS — FINNSENTIMENT-TUNED MODEL")

print("=" * 50)
print(classification_report(ft_true, fs_preds, target_names=LABEL_NAMES))

print("Confusion matrix (rows=true, cols=predicted):")
print(pd.DataFrame(
    confusion_matrix(ft_true, fs_preds),
    index=[f"true_{l}" for l in LABEL_NAMES],
    columns=[f"pred_{l}" for l in LABEL_NAMES],
))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

FinnSentiment-tuned model saved to: /content/drive/MyDrive/ColabThesisData/xlm-roberta-large_finnsentiment_2026-03-20_06-32-41/



HOLDOUT RESULTS — FINNSENTIMENT-TUNED MODEL
              precision    recall  f1-score   support

    negative       0.89      0.95      0.92       123
     neutral       0.89      0.83      0.86       123
    positive       0.92      0.92      0.92       136

    accuracy                           0.90       382
   macro avg       0.90      0.90      0.90       382
weighted avg       0.90      0.90      0.90       382

Confusion matrix (rows=true, cols=predicted):
               pred_negative  pred_neutral  pred_positive
true_negative            117             5              1
true_neutral              11           102             10
true_positive              4             7            125


## Preparing Human-labeled Financial Training Data

In [16]:
human_labeled_sentiment = pd.read_parquet('/content/drive/MyDrive/ColabThesisData/labeled.parquet')

# Replace with your dataset loading
ds = human_labeled_sentiment

In [17]:
ds.sample(5)

,id,author_id,message,date_time,engagement,forum,url,company_name,ticker,message_length,year_month,year,sentiment,labeled_at
109,Sijoitustieto.comment-422,Sijoitustieto.Unknown,"Nordea on kyllä hyvä pankki, mutta minua huole...",2014-07-22 13:41:00.000,N/A,Sijoitustieto,https://www.sijoitustieto.fi/sijoituskeskustel...,Nordea,NDA-FI,255,2014-07,2014,0,2026-03-16T17:18:44.966540
10,Kauppalehti.post-7315597,Kauppalehti.58646,Jos markkinoilla ois sama näkemys kuin tällä p...,2023-12-15 15:53:46.000,N/A,Kauppalehti,https://keskustelu.kauppalehti.fi/threads/ssh-...,SSH Communications Security,SSH1V,133,2023-12,2023,0,2026-03-16T09:39:30.502368
184,Inderes.313187,Inderes.6530,Kyllähän Harvian käyttökatteesta näkee että hi...,2021-07-04 04:44:54.112,38,Inderes,https://forum.inderes.com/t/harvia-foorumi-eli...,Harvia,HARVIA,206,2021-07,2021,1,2026-03-16T17:41:56.446339
77,Kauppalehti.post-7616837,Kauppalehti.57519,Arvuuttelija70 sanoi:\nTämä on tällaista... os...,2025-08-26 12:12:27.000,"Reactions:\nVerot, öölman ja Arvuuttelija70",Kauppalehti,https://keskustelu.kauppalehti.fi/threads/endo...,Endomines Finland,PAMPALO,661,2025-08,2025,1,2026-03-16T14:58:37.056246
538,Sijoitustieto.comment-600,Sijoitustieto.Unknown,Näinhän se on :)....mullakin suurin huoli siin...,2014-08-07 13:51:00.000,N/A,Sijoitustieto,https://www.sijoitustieto.fi/sijoituskeskustel...,UPM-Kymmene,UPM,176,2014-08,2014,1,2026-03-16T20:47:03.335281


In [18]:
ds["sentiment"].value_counts()

,count
sentiment,
1,253
2,205
0,150


## Fine-tune on Financial Domain Data

In [19]:
# 80/20 stratified split — holdout never seen during fine-tuning
ft_train_df, ft_holdout_df = train_test_split(
    ds[["message", "sentiment", "company_name"]],
    test_size=0.2,
    random_state=42,
    stratify=ds["sentiment"],
)

print(f"Fine-tune train: {len(ft_train_df)}  |  Holdout: {len(ft_holdout_df)}")
print("Train distribution:\n", ft_train_df["sentiment"].value_counts().sort_index())
print("Holdout distribution:\n", ft_holdout_df["sentiment"].value_counts().sort_index())

Fine-tune train: 486  |  Holdout: 122
Train distribution:
 sentiment
0    120
1    202
2    164
Name: count, dtype: int64
Holdout distribution:
 sentiment
0    30
1    51
2    41
Name: count, dtype: int64


In [20]:
def make_hf_dataset(df):
    df = df.copy()
    df["text"] = "yritys: " + df["company_name"] + " viesti: " + df["message"]
    hf = Dataset.from_pandas(df[["text", "sentiment"]].rename(columns={"sentiment": "label"}).reset_index(drop=True))
    return hf.map(
        lambda batch: tokenizer(batch["text"], truncation=True, max_length=MAX_SEQ_LENGTH),
        batched=True,
    )

ft_train_ds   = make_hf_dataset(ft_train_df)
ft_holdout_ds = make_hf_dataset(ft_holdout_df)

Map:   0%|          | 0/486 [00:00<?, ? examples/s]

Map:   0%|          | 0/122 [00:00<?, ? examples/s]

In [21]:
from torch import nn
from sklearn.utils.class_weight import compute_class_weight

model = AutoModelForSequenceClassification.from_pretrained(
    FINNSENTIMENT_MODEL_PATH,
    num_labels=NUM_LABELS,
    device_map="auto",
    dtype=torch.bfloat16,
)

FINETUNED_MODEL_PATH = f'/content/drive/MyDrive/ColabThesisData/{BASE_MODEL.split("/")[-1]}_finetuned_{timestamp}/'

# Compute class weights from the training split to counter neutral-class dominance
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=ft_train_df['sentiment'].values,
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(model.device)
print(f"Class weights: neg={class_weights[0]:.3f}  neu={class_weights[1]:.3f}  pos={class_weights[2]:.3f}")

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = nn.CrossEntropyLoss(weight=class_weights_tensor)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

ft_training_args = TrainingArguments(
    output_dir='/tmp/ft_checkpoints/',
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,           # 10% of total steps — more robust than a fixed count
    logging_steps=5,            # small dataset: log every 5 steps to see training loss
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    bf16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=True,
)

ft_trainer = WeightedTrainer(
    model=model,
    args=ft_training_args,
    train_dataset=ft_train_ds,
    eval_dataset=ft_holdout_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

ft_trainer.train()

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Class weights: neg=1.350  neu=0.802  pos=0.988


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,1.195072,1.024429,0.508197,0.469921,0.554375,0.508197
2,1.046555,0.977715,0.516393,0.495265,0.519128,0.516393
3,0.987081,0.990648,0.606557,0.594835,0.634438,0.606557
4,0.748642,0.966111,0.557377,0.544020,0.569057,0.557377
5,0.679048,0.956874,0.606557,0.604110,0.615000,0.606557
6,0.654420,0.963633,0.590164,0.588927,0.598394,0.590164
7,0.600470,0.999292,0.557377,0.548663,0.559788,0.557377
8,0.495501,0.991828,0.565574,0.561613,0.568689,0.565574
9,0.560450,0.993239,0.581967,0.579312,0.584595,0.581967
10,0.617172,0.995528,0.581967,0.578973,0.586843,0.581967


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=310, training_loss=0.7890193916136219, metrics={'train_runtime': 88.4026, 'train_samples_per_second': 54.976, 'train_steps_per_second': 3.507, 'total_flos': 3495289554750636.0, 'train_loss': 0.7890193916136219, 'epoch': 10.0})

In [22]:
# Save fine-tuned model
ft_trainer.save_model(FINETUNED_MODEL_PATH)
tokenizer.save_pretrained(FINETUNED_MODEL_PATH)
print(f"Fine-tuned model saved to: {FINETUNED_MODEL_PATH}")

# Evaluate on holdout
holdout_results = ft_trainer.predict(ft_holdout_ds)
ft_preds = np.argmax(holdout_results.predictions, axis=1)
ft_true = ft_holdout_df["sentiment"].tolist()

print("\n" + "=" * 50)
print("HOLDOUT RESULTS AFTER FINE-TUNING")
print("=" * 50)
print(classification_report(ft_true, ft_preds, target_names=LABEL_NAMES))

print("Confusion matrix (rows=true, cols=predicted):")
print(pd.DataFrame(
    confusion_matrix(ft_true, ft_preds),
    index=[f"true_{l}" for l in LABEL_NAMES],
    columns=[f"pred_{l}" for l in LABEL_NAMES],
))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned model saved to: /content/drive/MyDrive/ColabThesisData/xlm-roberta-large_finetuned_2026-03-20_06-32-41/



HOLDOUT RESULTS AFTER FINE-TUNING
              precision    recall  f1-score   support

    negative       0.56      0.63      0.59        30
     neutral       0.67      0.51      0.58        51
    positive       0.59      0.71      0.64        41

    accuracy                           0.61       122
   macro avg       0.61      0.62      0.61       122
weighted avg       0.62      0.61      0.60       122

Confusion matrix (rows=true, cols=predicted):
               pred_negative  pred_neutral  pred_positive
true_negative             19             5              6
true_neutral              11            26             14
true_positive              4             8             29


## Augmented Training: Human + LLM Labels

In [23]:
LLM_LABELS_PATH = '/content/drive/MyDrive/ColabThesisData/llm_labeled.parquet'

llm_labels = pd.read_parquet(LLM_LABELS_PATH)

# Safety: recover holdout IDs from the original ds index and exclude them
holdout_ids = set(ds.loc[ft_holdout_df.index, 'id'])
leaked = llm_labels[llm_labels['id'].isin(holdout_ids)]
if len(leaked):
    print(f"WARNING: {len(leaked)} LLM labels overlap with holdout — dropping them.")
    llm_labels = llm_labels[~llm_labels['id'].isin(holdout_ids)]

# Combine: human train split + all LLM labels (keep company_name for text formatting)
ft_train_augmented_df = pd.concat(
    [ft_train_df, llm_labels[["message", "sentiment", "company_name"]]],
    ignore_index=True,
)

print(f"Train  : {len(ft_train_df)} human  →  {len(ft_train_augmented_df)} total (+{len(llm_labels)} LLM)")
print(f"Holdout: {len(ft_holdout_df)} (unchanged)")
print("\nAugmented train sentiment distribution:")
print(ft_train_augmented_df["sentiment"].value_counts().sort_index().rename({0: "negative", 1: "neutral", 2: "positive"}))

Train  : 486 human  →  10486 total (+10000 LLM)
Holdout: 122 (unchanged)

Augmented train sentiment distribution:
sentiment
negative    3711
neutral     2984
positive    3791
Name: count, dtype: int64


In [24]:
# Fine tuned weights as new base — should converge faster and handle more data better than starting from the original base model
model_aug = AutoModelForSequenceClassification.from_pretrained(
    FINETUNED_MODEL_PATH,
    num_labels=NUM_LABELS,
    device_map="auto",
    dtype=torch.bfloat16,
)

ft_train_augmented_ds = make_hf_dataset(ft_train_augmented_df)
# ft_holdout_ds is unchanged

AUGMENTED_MODEL_PATH = f'/content/drive/MyDrive/ColabThesisData/{BASE_MODEL.split("/")[-1]}_augmented_{timestamp}/'

aug_training_args = TrainingArguments(
    output_dir='/tmp/aug_checkpoints/',
    num_train_epochs=15,         # more data needs more epochs to converge
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=3e-5,          # slightly higher — larger dataset can support it
    weight_decay=0.01,
    warmup_steps=250,
    logging_steps=50,            # log training loss every 50 steps so we can see it move
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    bf16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=True,
)

aug_trainer = Trainer(
    model=model_aug,
    args=aug_training_args,
    train_dataset=ft_train_augmented_ds,
    eval_dataset=ft_holdout_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

aug_trainer.train()

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Map:   0%|          | 0/10486 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.807499,0.940412,0.581967,0.564220,0.664646,0.581967
2,0.718940,0.871802,0.663934,0.659976,0.696104,0.663934
3,0.659687,0.924574,0.655738,0.652604,0.728333,0.655738
4,0.613051,0.966365,0.639344,0.629439,0.678308,0.639344
5,0.486304,1.026406,0.672131,0.671680,0.714652,0.672131
6,0.429691,1.085991,0.663934,0.664819,0.705744,0.663934
7,0.401108,1.012952,0.696721,0.698599,0.726756,0.696721
8,0.421542,1.159891,0.655738,0.654838,0.701869,0.655738
9,0.344311,1.208722,0.647541,0.641464,0.697380,0.647541
10,0.335948,1.224721,0.655738,0.652668,0.707384,0.655738


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=9840, training_loss=0.45771920147950085, metrics={'train_runtime': 730.6824, 'train_samples_per_second': 215.265, 'train_steps_per_second': 13.467, 'total_flos': 1.0814022828002709e+17, 'train_loss': 0.45771920147950085, 'epoch': 15.0})

In [25]:
aug_trainer.save_model(AUGMENTED_MODEL_PATH)
tokenizer.save_pretrained(AUGMENTED_MODEL_PATH)
print(f"Augmented model saved to: {AUGMENTED_MODEL_PATH}")

aug_results = aug_trainer.predict(ft_holdout_ds)
aug_preds = np.argmax(aug_results.predictions, axis=1)
ft_true   = ft_holdout_df["sentiment"].tolist()

print("\n" + "=" * 50)
print("HOLDOUT RESULTS — AUGMENTED (human + LLM)")
print("=" * 50)
print(classification_report(ft_true, aug_preds, target_names=LABEL_NAMES))

print("Confusion matrix (rows=true, cols=predicted):")
print(pd.DataFrame(
    confusion_matrix(ft_true, aug_preds),
    index=[f"true_{l}" for l in LABEL_NAMES],
    columns=[f"pred_{l}" for l in LABEL_NAMES],
))

# Side-by-side summary vs the human-only run
print("\n── Accuracy comparison ──")
print(f"  Human only  (~486 train) : {accuracy_score(ft_true, ft_preds):.3f}")
print(f"  Human + LLM (~{len(ft_train_augmented_df)} train) : {accuracy_score(ft_true, aug_preds):.3f}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Augmented model saved to: /content/drive/MyDrive/ColabThesisData/xlm-roberta-large_augmented_2026-03-20_06-32-41/



HOLDOUT RESULTS — AUGMENTED (human + LLM)
              precision    recall  f1-score   support

    negative       0.54      0.73      0.62        30
     neutral       0.83      0.59      0.69        51
    positive       0.73      0.80      0.77        41

    accuracy                           0.70       122
   macro avg       0.70      0.71      0.69       122
weighted avg       0.73      0.70      0.70       122

Confusion matrix (rows=true, cols=predicted):
               pred_negative  pred_neutral  pred_positive
true_negative             22             5              3
true_neutral              12            30              9
true_positive              7             1             33

── Accuracy comparison ──
  Human only  (~486 train) : 0.607
  Human + LLM (~10486 train) : 0.697
